# 🚀 ConveyorVision — YOLO Training & Visual Analytics

Notebook huấn luyện mô hình YOLO phân loại sản phẩm đạt (`pass`) và lỗi (`defect`) trên băng chuyền.
- Hỗ trợ chạy trực tiếp trên **Google Colab GPU** (thông qua Colab Extension) hoặc **môi trường cục bộ**.
- Tích hợp biểu đồ trực quan hóa tiến trình học (Loss, Precision, Recall, mAP, Confusion Matrix) và hình ảnh dự đoán thực tế.

## 1. Kiểm tra môi trường & Phần cứng (GPU/CPU)

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("🔥 GPU Device:", torch.cuda.get_device_name(0))
    print("Bộ nhớ GPU khả dụng:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("⚡ Đang chạy trên CPU")

## 2. Cài đặt các thư viện cần thiết

In [ ]:
%pip install -q ultralytics matplotlib seaborn pandas pillow roboflow

## 3. Đồng bộ Dữ liệu (Dành cho Google Colab Remote Kernel)
*Nếu chạy trên máy cục bộ, bạn có thể bỏ qua cell này vì thư mục `data/` đã có sẵn 2.666 ảnh.*

In [ ]:
import os
from pathlib import Path

# Kiểm tra nếu đang chạy trên Colab
if not Path("data").exists():
    print(">>> Đang tải mã nguồn từ GitHub...")
    !git clone https://github.com/anky06-ky/ConveyorVision.git
    %cd ConveyorVision
    
    # Tải các bộ dữ liệu qua Roboflow API
    ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"
    !python src/conveyorvision/dataset_manager.py --download-all --api-key {ROBOFLOW_API_KEY}
else:
    print("✅ Thư mục dữ liệu 'data/' đã sẵn sàng!")

## 4. Huấn Luyện Mô Hình YOLO (30 Epochs)
Với GPU trên Colab, quá trình huấn luyện 30 epochs sẽ hoàn thành trong khoảng **10 - 15 phút**!

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình pretrained nano
model = YOLO('yolov8n.pt')

# Huấn luyện mô hình 30 epochs
results = model.train(
    data='data/data.yaml',
    epochs=30,
    batch=16,
    imgsz=640,
    project='runs/train',
    name='conveyor_yolo30',
    plots=True,
    save=True,
    verbose=True
)

## 5. Trực Quan Hóa Biểu Đồ & Kết Quả Huấn Luyện
Hiển thị trực quan các đường cong Loss, mAP, Precision-Recall và Ma trận nhầm lẫn (Confusion Matrix).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Tìm thư mục kết quả huấn luyện mới nhất hoặc pilot_test
exp_dirs = list(Path('runs/detect/runs/pilot').glob('*')) + list(Path('runs/train').glob('*'))
latest_dir = exp_dirs[-1] if exp_dirs else Path('runs/detect/runs/pilot/pilot_test')
print(f"Hiển thị kết quả từ thư mục: {latest_dir}")

# 1. Biểu đồ tiến trình huấn luyện (Loss & Metrics)
results_img = latest_dir / 'results.png'
if results_img.exists():
    plt.figure(figsize=(14, 8))
    plt.imshow(Image.open(results_img))
    plt.title('Tiến trình huấn luyện (Loss, Precision, Recall, mAP)', fontsize=14, pad=10)
    plt.axis('off')
    plt.show()

# 2. Ma trận nhầm lẫn (Confusion Matrix)
cm_img = latest_dir / 'confusion_matrix.png'
if cm_img.exists():
    plt.figure(figsize=(10, 8))
    plt.imshow(Image.open(cm_img))
    plt.title('Ma trận nhầm lẫn (Confusion Matrix)', fontsize=14, pad=10)
    plt.axis('off')
    plt.show()

# 3. Đường cong Precision - Recall (PR Curve)
pr_img = latest_dir / 'BoxPR_curve.png'
if pr_img.exists():
    plt.figure(figsize=(10, 7))
    plt.imshow(Image.open(pr_img))
    plt.title('Đường cong Precision - Recall', fontsize=14, pad=10)
    plt.axis('off')
    plt.show()

## 6. Trực Quan Hóa Ảnh Dự Đoán Thực Tế (Inference Preview)
Xem cách mô hình bắt bounding box và phân biệt giữa chai Đạt (`pass`) và chai Lỗi (`defect`).

In [ ]:
# Hiển thị ảnh dự đoán trên tập validation
pred_imgs = list(latest_dir.glob('val_batch*_pred.jpg'))
if pred_imgs:
    fig, axes = plt.subplots(1, min(2, len(pred_imgs)), figsize=(18, 9))
    if not isinstance(axes, (list, tuple)) and not hasattr(axes, '__iter__'):
        axes = [axes]
    for ax, p_img in zip(axes, pred_imgs[:2]):
        ax.imshow(Image.open(p_img))
        ax.set_title(p_img.name, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Chưa tìm thấy ảnh dự đoán trong thư mục kết quả.")